## Data

This project uses Earnings-21, a set of 44 English earnings calls with human reference transcripts, loaded from the Hugging Face ASR leaderboard long-form collection. Earnings calls are a realistic captioning use case since they mix multiple speakers, numbers, company names and financial terms.

The analysis uses the first two of the ten data files, which hold 10 calls and about 10.75 hours of audio across six sectors. TED-LIUM was the original choice, but it is no longer available through Hugging Face.

In [2]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset

# load the two earnings-21 files that already finished downloading
repo = "hf-audio/asr-leaderboard-longform"
names = ["earnings21/test-00000-of-00010.parquet", "earnings21/test-00001-of-00010.parquet"]
paths = [hf_hub_download(repo, n, repo_type="dataset") for n in names]

ds = load_dataset("parquet", data_files=paths, split="train")

print(ds)
print()

for s in ds.remove_columns("audio"):
    print(s["company_name"], s["sector"], s["unique_speakers"], round(s["audio_length"] / 60), "min")

Dataset({
    features: ['audio', 'audio_length', 'original_sample_rate', 'company_name', 'financial_quarter', 'sector', 'speaker_switches', 'unique_speakers', 'curator_id', 'text'],
    num_rows: 10
})

Monro Inc Consumer Goods 10 55 min
Culp Inc Industrial Goods 8 41 min
General Electric Conglomerate 14 96 min
Danaher Corp Conglomerate 7 45 min
Spire Inc Utilities 10 55 min
Ingersoll Rand Industrial Goods 14 66 min
Cementos Argos Industrial Goods 20 78 min
Kuehne Nagel International Services 13 81 min
Constellium Industrial Goods 10 63 min
Travelers Companies Inc Financial 15 65 min


## Transcription test

Before transcribing all 10 calls, this cell runs Whisper on the first two minutes of the Culp Inc call, which is the shortest in the subset. It checks how long transcription takes on a Mac and gives a first look at how Whisper’s output compares with the human reference.

In [3]:
import time
import whisper
from datasets import Audio

# get the culp call as raw audio bytes so whisper can decode it with ffmpeg
ds_raw = ds.cast_column("audio", Audio(decode=False))
idx = list(ds["company_name"]).index("Culp Inc")
culp = ds_raw[idx]

with open("culp.wav", "wb") as f:
    f.write(culp["audio"]["bytes"])

# load the audio at 16 kHz and keep the first two minutes
audio = whisper.load_audio("culp.wav")
clip = audio[: 16000 * 120]

model = whisper.load_model("small.en")

start = time.time()
result = model.transcribe(clip, fp16=False)
elapsed = time.time() - start

print(round(elapsed), "seconds to transcribe 2 minutes of audio")
print(round(elapsed / 2 * 645 / 60, 1), "estimated hours for all 10 calls")
print()
print(result["text"])
print()
print(culp["text"][:1500])

100%|███████████████████████████████████████| 461M/461M [01:03<00:00, 7.63MiB/s]


15 seconds to transcribe 2 minutes of audio
81.8 estimated hours for all 10 calls

 Good day and welcome to Culp's third quarter 2020 earnings conference call. Today's call is being recorded at this time for opening remarks and introductions. I'd like to turn the call over to Ms. Drew Anderson. Please go ahead. Thank you. Good morning and welcome to the Culp conference call to review the company's results for the third quarter of fiscal 2020. As we start, let me state that this morning's call will contain forward-looking statements about the business, financial condition, and prospects of the company. Forward-looking statements are statements that include projections, expectations, or beliefs about future events or results, or otherwise are not statements of historical fact. The actual performance of the company could differ materially from that indicated by the forward-looking statements because of various risks and uncertainties. These risks and uncertainties are described in our reg

## Transcribing all calls

The test showed that the small.en model transcribes about 8 times faster than real time, so all 10 calls take roughly 78 minutes. That makes small.en a good balance of speed and accuracy for this project.

Each finished call is saved to whisper_transcripts.csv along with its reference transcript and metadata. If the run is interrupted, rerunning the cell skips calls that are already saved.

In [5]:
import os
import pandas as pd

# load any transcripts saved from an earlier run so finished calls are skipped
rows = []
if os.path.exists("whisper_transcripts.csv"):
    rows = pd.read_csv("whisper_transcripts.csv").to_dict("records")
done = {r["company_name"] for r in rows}

for i in range(len(ds_raw)):
    call = ds_raw[i]
    name = call["company_name"]
    if name in done:
        continue

    with open("call.wav", "wb") as f:
        f.write(call["audio"]["bytes"])
    audio = whisper.load_audio("call.wav")

    start = time.time()
    result = model.transcribe(audio, fp16=False)
    minutes = round((time.time() - start) / 60, 1)

    rows.append({
        "company_name": name,
        "sector": call["sector"],
        "unique_speakers": call["unique_speakers"],
        "speaker_switches": call["speaker_switches"],
        "audio_minutes": round(call["audio_length"] / 60, 1),
        "transcribe_minutes": minutes,
        "whisper_text": result["text"],
        "reference_text": call["text"],
    })

    # save after every call so progress is never lost
    pd.DataFrame(rows).to_csv("whisper_transcripts.csv", index=False)
    print(name, "done in", minutes, "minutes")

Monro Inc done in 7.0 minutes
Culp Inc done in 5.6 minutes
General Electric done in 11.8 minutes
Danaher Corp done in 5.5 minutes
Spire Inc done in 7.0 minutes
Ingersoll Rand done in 8.6 minutes
Cementos Argos done in 8.1 minutes
Kuehne Nagel International done in 9.0 minutes
Constellium done in 7.7 minutes
Travelers Companies Inc done in 7.9 minutes


## Word error rate

Word error rate (WER) counts the substituted, deleted and inserted words needed to turn Whisper's transcript into the reference, divided by the number of reference words. Both transcripts are normalized first so that case and punctuation are not counted as errors.

WER is computed two ways. **Light normalization** only lowercases and removes punctuation. **Full normalization** uses Whisper's English normalizer, which also standardizes numbers, spellings and contractions and removes filler words like "um" and "uh." Full normalization is the standard used by the Open ASR Leaderboard, so these results can be compared with published numbers.

Overall WER is 14.8% with light normalization and 11.8% with full normalization. The gap shows that about a fifth of the apparent errors come from formatting differences and not recognition mistakes. Removing filler words also means these results treat clean verbatim captions as the standard.

In [6]:
import re
import jiwer
from whisper.normalizers import EnglishTextNormalizer

df = pd.read_csv("whisper_transcripts.csv")
normalizer = EnglishTextNormalizer()

# light normalization lowercases and removes punctuation but keeps word forms
def light_normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s']", " ", text)
    return " ".join(text.split())

df["ref_light"] = df["reference_text"].apply(light_normalize)
df["hyp_light"] = df["whisper_text"].apply(light_normalize)
df["ref_full"] = df["reference_text"].apply(normalizer)
df["hyp_full"] = df["whisper_text"].apply(normalizer)

df["wer_light"] = [jiwer.wer(r, h) for r, h in zip(df["ref_light"], df["hyp_light"])]
df["wer_full"] = [jiwer.wer(r, h) for r, h in zip(df["ref_full"], df["hyp_full"])]

print(df[["company_name", "sector", "unique_speakers", "wer_light", "wer_full"]].round(3))
print()
print("overall light WER", round(jiwer.wer(list(df["ref_light"]), list(df["hyp_light"])), 3))
print("overall full WER", round(jiwer.wer(list(df["ref_full"]), list(df["hyp_full"])), 3))

                 company_name            sector  unique_speakers  wer_light  \
0                   Monro Inc    Consumer Goods               10      0.109   
1                    Culp Inc  Industrial Goods                8      0.095   
2            General Electric      Conglomerate               14      0.144   
3                Danaher Corp      Conglomerate                7      0.135   
4                   Spire Inc         Utilities               10      0.104   
5              Ingersoll Rand  Industrial Goods               14      0.195   
6              Cementos Argos  Industrial Goods               20      0.268   
7  Kuehne Nagel International          Services               13      0.164   
8                 Constellium  Industrial Goods               10      0.119   
9     Travelers Companies Inc         Financial               15      0.106   

   wer_full  
0     0.085  
1     0.080  
2     0.122  
3     0.094  
4     0.081  
5     0.170  
6     0.239  
7     0.101  
8   

## Types of errors

WER combines three kinds of errors. **Substitutions** are wrong words, **deletions** are missed words and **insertions** are added words. This cell counts each type per call and lists the most common substituted word pairs across all calls.

The substitutions fall into four groups.

- **Formatting differences**, such as "cashflow" in the reference and "cash flow" in Whisper's output
- **Reference errors where Whisper is correct**, such as "Columbia" for the Colombian company Cementos Argos
- **Proper noun errors**, where Whisper uses a more common spelling of an unfamiliar name, such as "Monroe" for Monro and "Paul" for Pall
- **Short unstressed words and grammatical forms**, such as "and" and "in" or "were" and "are"

In [7]:
from collections import Counter

# split each call's errors into substitutions deletions and insertions
# and collect every substituted word pair across all calls
counts = []
swaps = Counter()

for _, row in df.iterrows():
    out = jiwer.process_words(row["ref_full"], row["hyp_full"])
    ref, hyp = out.references[0], out.hypotheses[0]

    counts.append({
        "company_name": row["company_name"],
        "ref_words": len(ref),
        "substitutions": out.substitutions,
        "deletions": out.deletions,
        "insertions": out.insertions,
    })

    for chunk in out.alignments[0]:
        if chunk.type == "substitute":
            ref_span = ref[chunk.ref_start_idx:chunk.ref_end_idx]
            hyp_span = hyp[chunk.hyp_start_idx:chunk.hyp_end_idx]
            for r, h in zip(ref_span, hyp_span):
                swaps[(r, h)] += 1

err = pd.DataFrame(counts)
print(err.to_string(index=False))
print()

for (r, h), n in swaps.most_common(30):
    print(n, r, "->", h)

              company_name  ref_words  substitutions  deletions  insertions
                 Monro Inc       8675            313        340          81
                  Culp Inc       6639            239        209          86
          General Electric      14643            532       1018         239
              Danaher Corp       6946            225        371          58
                 Spire Inc       9133            236        412          90
            Ingersoll Rand      10996            673       1044         148
            Cementos Argos      10157           1177        860         388
Kuehne Nagel International      11576            440        601         130
               Constellium       9604            290        455          83
   Travelers Companies Inc      10499            319        514          98

38 and -> in
37 in -> and
26 monro -> monroe
24 the -> a
24 a -> the
22 these -> this
19 costs -> cost
15 cashflow -> flow
15 202 -> 2020
15 columbia -> colombia
1

## Checking errors in context

This function prints the words around each instance of a substitution in both transcripts, which makes it possible to tell real recognition errors apart from formatting or reference problems.

In [8]:
# show the surrounding words for each instance of a reference to whisper swap
def show_swap(ref_word, hyp_word, window=6, limit=5):
    shown = 0
    for _, row in df.iterrows():
        out = jiwer.process_words(row["ref_full"], row["hyp_full"])
        ref, hyp = out.references[0], out.hypotheses[0]
        for chunk in out.alignments[0]:
            if chunk.type != "substitute":
                continue
            for k in range(chunk.ref_end_idx - chunk.ref_start_idx):
                ri = chunk.ref_start_idx + k
                hi = chunk.hyp_start_idx + k
                if ref[ri] == ref_word and hyp[hi] == hyp_word:
                    print(row["company_name"])
                    print("ref ", " ".join(ref[max(0, ri - window):ri + window]))
                    print("hyp ", " ".join(hyp[max(0, hi - window):hi + window]))
                    print()
                    shown += 1
                    if shown >= limit:
                        return

show_swap("202", "2020")
show_swap("cashflow", "flow")
show_swap("and", "in")

Spire Inc
ref  the 2nd half of our fiscal 202 turning to our results for
hyp  the 2nd half of our fiscal 2020 turning to our results for

Spire Inc
ref  equity needs for the remainder of 202 so in summary we are
hyp  equity needs for the remainder of 2020 so in summary we are

Kuehne Nagel International
ref  419000000 in the 1st half of 202 earnings per share landed at
hyp  francs in the 1st half of 2020 earnings per share landed at

Kuehne Nagel International
ref  more than tripled in quarter 2 202 road logistics ended the 1st
hyp  more than tripled in quarter to 2020 road logistics ended the 1st

Kuehne Nagel International
ref  the end of the 1st quarter 202 so simple terms a very
hyp  the end of the 1st quarter 2020 simple terms a very good

Culp Inc
ref  respect to certain forward looking free cashflow information the comparable gaap and
hyp  to certain forward looking free cash flow information the comparable gaap and

Culp Inc
ref  the significance of the historical free cashflow

## Investigating the year errors

The normalized reference contained "202" in several places where Whisper wrote "2020." These cells check how years are written in the raw reference text and then look at the raw text at two of those spots.

The raw reference itself says "202," so these are typos made by the original transcriber and not a normalizer problem. The "201" errors follow the same pattern. In these cases WER counted Whisper as wrong even though it heard the year correctly.

In [10]:
from collections import Counter

# collect every token in the raw references that contains a year
year_forms = Counter()
for text in df["reference_text"]:
    year_forms.update(re.findall(r"\S*20[12]\d\S*", text))

for form, n in year_forms.most_common(30):
    print(n, repr(form))

63 '2020'
25 '2020,'
17 '2020.'
14 '2019.'
12 '2019,'
12 '2019'
9 '2021,'
9 '2021'
3 '2021.'
3 '2015'
2 '2016'
2 '2018.'
2 '2019?'
2 '2018'
2 '2022,'
1 '2017'
1 '2022'
1 '2023.'
1 "2020's,"
1 '2016,'
1 '2024'
1 '2018,'
1 '2027.'
1 '2020?'
1 '2028,'
1 '2025'
1 '2013'


In [11]:
# pull the raw reference text where the 202 problem appeared and normalize it
spire = df.loc[df["company_name"] == "Spire Inc", "reference_text"].iloc[0]
kuehne = df.loc[df["company_name"] == "Kuehne Nagel International", "reference_text"].iloc[0]

for text, phrase in [(spire, "remainder of"), (kuehne, "end of the first quarter")]:
    i = text.find(phrase)
    snippet = text[i:i + 70]
    print(repr(snippet))
    print(normalizer(snippet))
    print()

"remainder of 202. So in summary, we're in solid shape and remain focus"
remainder of 202 so in summary we are in solid shape and remain focus

'end of the first quarter 202. So, um, simple terms, a very good, uh, u'
end of the 1st quarter 202 so simple terms a very good u



## Cleaning up conventions and reference errors

This step applies two small sets of fixes and recomputes WER.

- **Convention fixes** are applied to both transcripts so that spacing choices like "cashflow" and "cash flow" match.
- **Reference fixes** are applied only to the reference and correct typos confirmed in the raw text. "Columbia" is corrected only in the Cementos Argos call, where it refers to the country.

The cleanup lowers overall WER only slightly, from 11.8% to 11.6%. Reference typos and spacing conventions are real but minor, and nearly all of the remaining error is genuine recognition error. Cementos Argos still has the highest WER at 23.6%, so its high error rate needs another explanation.

In [12]:
# spacing conventions that differ between transcripts and are not real errors
convention_fixes = {"cashflow": "cash flow", "healthcare": "health care", "longterm": "long term", "u s": "us"}

# typos in the reference transcripts confirmed by checking the raw text
reference_fixes = {"202": "2020", "201": "2019"}

def apply_fixes(text, fixes):
    for old, new in fixes.items():
        text = re.sub(rf"\b{re.escape(old)}\b", new, text)
    return text

df["ref_clean"] = df["ref_full"].apply(apply_fixes, fixes=convention_fixes).apply(apply_fixes, fixes=reference_fixes)
df["hyp_clean"] = df["hyp_full"].apply(apply_fixes, fixes=convention_fixes)

# colombia is only fixed in the cementos argos call where it refers to the country
is_argos = df["company_name"] == "Cementos Argos"
df.loc[is_argos, "ref_clean"] = df.loc[is_argos, "ref_clean"].str.replace(r"\bcolumbia\b", "colombia", regex=True)

df["wer_clean"] = [jiwer.wer(r, h) for r, h in zip(df["ref_clean"], df["hyp_clean"])]

print(df[["company_name", "wer_light", "wer_full", "wer_clean"]].round(3).to_string(index=False))
print()
print("overall light WER", round(jiwer.wer(list(df["ref_light"]), list(df["hyp_light"])), 3))
print("overall full WER", round(jiwer.wer(list(df["ref_full"]), list(df["hyp_full"])), 3))
print("overall clean WER", round(jiwer.wer(list(df["ref_clean"]), list(df["hyp_clean"])), 3))

              company_name  wer_light  wer_full  wer_clean
                 Monro Inc      0.109     0.085      0.084
                  Culp Inc      0.095     0.080      0.078
          General Electric      0.144     0.122      0.120
              Danaher Corp      0.135     0.094      0.094
                 Spire Inc      0.104     0.081      0.079
            Ingersoll Rand      0.195     0.170      0.169
            Cementos Argos      0.268     0.239      0.236
Kuehne Nagel International      0.164     0.101      0.100
               Constellium      0.119     0.086      0.083
   Travelers Companies Inc      0.106     0.089      0.087

overall light WER 0.148
overall full WER 0.118
overall clean WER 0.116
